# Inferência Baseline - Tweet Sentiment Analysis

Este notebook avalia o desempenho do modelo pré-treinado **CardiffNLP Twitter-RoBERTa Base Sentiment** diretamente no dataset de teste, sem fine-tuning, para estabelecer uma linha de base (baseline).

## Objetivo

Responder às seguintes perguntas-chave antes de realizar fine-tuning:

1. **Desempenho Zero-Shot**: Qual o desempenho do modelo pré-treinado sem qualquer ajuste?
2. **Erros por Classe**: Quais classes o modelo confunde com mais frequência?
3. **Métricas de Referência**: Qual o F1-score, precisão e recall como baseline?

---
## Setup

Importação das bibliotecas necessárias para inferência e avaliação.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datasets as dts
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

---
## Carregamento do Modelo e Dados

Carregamos o modelo pré-treinado `twitter-roberta-base-sentiment` e o dataset `tweet_eval` (subset sentiment) para avaliação direta sem fine-tuning.

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

# Carrega modelo pré-treinado e tokenizer
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Carrega dataset com splits train/validation/test
dataset = dts.load_dataset("tweet_eval", "sentiment")

---
## Tokenização do Dataset de Teste

Aplicamos o tokenizer no split de teste com `max_length=64`, definido na análise de tokenização anterior.

In [ ]:
def prepare_data(example):
    """Tokeniza o texto com padding e truncamento."""
    return tokenizer(example["text"], padding="max_length", max_length=64, truncation=True)


# Aplica tokenização em todo o split de teste
tokenized_test = dataset["test"].map(prepare_data)

---
## Inferência com o Modelo Baseline

Realizamos predições no dataset de teste utilizando o `Trainer` da Hugging Face, sem treinamento prévio, para avaliar o desempenho zero-shot do modelo.

In [ ]:
train_args = TrainingArguments(output_dir="./baseline")
trainer = Trainer(model=model, args=train_args, eval_dataset=tokenized_test)

# Seleciona subconjunto de 1000 amostras para avaliação controlada
subset = tokenized_test.select(range(1000))
result = trainer.predict(subset)

# Formato: (n_amostras, n_classes)
print(result.predictions.shape)

---
## Avaliação dos Resultados

Analisamos a qualidade das predições através da matriz de confusão e do relatório de classificação com métricas por classe.

In [ ]:
# Classe predita = índice com maior probabilidade
y_pred = np.argmax(result.predictions, axis=1)
y_true = result.label_ids

# Matriz de confusão: linhas = real, colunas = predito
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Valores Preditos")
plt.ylabel("Valores Reais")
plt.title("Matriz de Confusão - Baseline")
plt.show()

# Relatório com precision, recall e F1-score por classe
print(classification_report(y_true, y_pred))

### Conclusão - Avaliação do Baseline

O modelo pré-treinado sem fine-tuning obteve **accuracy de 70%** e **macro F1-score de 0.71** na amostra de 1000 exemplos do split de teste.

#### Métricas por Classe

| Classe | Precision | Recall | F1-Score | Support |
|--------|-----------|--------|----------|---------|
| **0 - Negative** | 0.64 | 0.76 | 0.70 | 304 |
| **1 - Neutral** | 0.73 | 0.67 | 0.70 | 499 |
| **2 - Positive** | 0.76 | 0.71 | 0.73 | 197 |
| **Macro Avg** | 0.71 | 0.71 | 0.71 | 1000 |
| **Weighted Avg** | 0.71 | 0.70 | 0.71 | 1000 |

**Observações:**
- **Negative** tem o menor precision (0.64) mas o maior recall (0.76) — o modelo tende a classificar demais como negativo
- **Neutral** tem o menor recall (0.67) — sendo a classe majoritária, parte dos neutros é confundida com as demais classes
- **Positive** apresenta o melhor F1-score (0.73), indicando que o modelo distingue bem sentimentos positivos
- Erros concentrados entre classes adjacentes (negative/neutral e neutral/positive) são esperados por proximidade semântica

---
## Resumo e Próximos Passos

### Principais Descobertas

1. **Baseline estabelecido**: Accuracy de 70% e macro F1-score de 0.71 sem fine-tuning
2. **Padrões de erro**: Classe neutral é a mais confundida, negative tende a ser sobre-predita
3. **Margem para melhoria**: Fine-tuning deve melhorar significativamente as métricas

### Recomendações para Modelagem

- [ ] Realizar fine-tuning no split de treino com class weights
- [ ] Comparar métricas pós-treino com este baseline (F1 macro > 0.71)
- [ ] Avaliar impacto de diferentes `max_length` e learning rates
- [ ] Usar **macro F1-score** como métrica principal (dataset desbalanceado)